# Protein–Ligand MD with GROMACS: from a Windows laptop to a 100 ns trajectory, analysis and MM/GBSA

**A complete teaching guide (version 2).** It follows one real project, a COX–diclofenac complex (CHARMM27 force field, TIP3P water, about 61,590 atoms, 100 ns), from first setting up the laptop to the final binding-energy calculation. It also records the problems we actually ran into, and how each was fixed.

### How to read this notebook
- **Grey code boxes inside text cells** are commands to run **on your own computer**: PowerShell for Windows commands, the Ubuntu terminal for Linux commands. Paste **one line at a time**, because multi-line pastes often get garbled in the Windows terminal.
- **Code cells with a play button** (Parts 7–9) are meant to run in **Google Colab**.
- **Group numbers** (for example `4` for Backbone, `13` for LIG) belong to our example system. Always read the list GROMACS prints and choose the numbers that match *your* system.
- Names such as `COX-diclo`, `REC.pdb` and `LIG.mol2` are our example files. Replace them with your own.

### Where each stage runs

| Stage | Where | Tool | GPU needed? |
|---|---|---|---|
| 1. Ligand and receptor preparation | Windows | Chimera, SwissParam | No |
| 2. Build the system, add ions, energy minimization | Ubuntu (WSL2) | GROMACS (CPU) | No |
| 3. NVT, NPT and the 100 ns production run | Google Colab | GROMACS (CUDA build) | **Yes** |
| 4. Trajectory processing and analysis | Ubuntu (WSL2) | GROMACS (CPU) | No |
| 5. MM/GBSA binding energy | Ubuntu (WSL2) | gmx_MMPBSA | No |

### Real timings from this project (use them to plan)

| Task | Hardware | Time |
|---|---|---|
| NVT, 100 ps | Colab L4 GPU | about 94 s (about 92 ns/day), measured |
| Production, 100 ns | Colab L4 GPU | about 14 h 52 min (about 161 ns/day), measured |
| Production, CPU only | i7-7500U laptop, 2 cores | roughly 0.3–1 ns/day, an *estimate*, not measured |
| MM/GBSA, 201 frames | same laptop | about 59 min, measured |

Everything before production (Parts 1–6) and everything after it (Parts 10–12) runs comfortably on an ordinary laptop. Only the long production run needs a GPU.

---
## Part 1 - Windows: enable virtualization and install WSL2 + Ubuntu

*Run in: BIOS and Windows PowerShell (as Administrator).*

WSL2 needs hardware virtualization. If it is switched off, the install fails with:

```
WslRegisterDistribution failed with error: 0x80370102
Please enable the Virtual Machine Platform Windows feature and ensure virtualization is enabled in the BIOS.
```

### 1.1 Enable the Windows features
```
dism.exe /online /enable-feature /featurename:VirtualMachinePlatform /all /norestart
dism.exe /online /enable-feature /featurename:Microsoft-Windows-Subsystem-Linux /all /norestart
```
Restart Windows afterwards.

### 1.2 Check virtualization
Task Manager → **Performance** → **CPU**. At the bottom right, read the **Virtualization** line. (The *Details* tab does not show it.) If it says **Disabled**, enable it in the BIOS even when the CPU supports it ("Hyper-V support: Yes").

### 1.3 Enable it in the BIOS
1. Restart and tap **Esc** repeatedly at the logo (HP laptops), then press **F10** for BIOS Setup. Other makers use F2, F10 or Del.
2. Go to **Advanced → System Options** (the name varies).
3. Enable **Virtualization Technology (VTx)**, and **VTd** if present.
4. Press **F10**, confirm **Yes**, and let the machine reboot.
5. Re-check Task Manager. It should now say **Enabled**.

> On a company-managed laptop the BIOS may be password-protected. IT will then need to enable it.

### 1.4 Install Ubuntu
```
wsl --install -d Ubuntu
```
On first launch, Ubuntu asks for a Unix username and password. **Nothing appears on screen while you type the password.** That is normal. It then asks whether to share anonymous usage data (optional). When you see `username@DESKTOP-XXXX:~$`, the install is complete.

**Persistence:** closing the Ubuntu window or restarting Windows does *not* erase anything you installed or saved. A command that is *running* (such as a simulation) is stopped when you close the window.

### Troubleshooting the WSL install

**Stuck at "Waiting for OOBE command to complete"** on a brand-new install. Reset it, and only do this when nothing has been saved in Ubuntu yet:
```
wsl --shutdown
wsl --unregister Ubuntu
wsl --install -d Ubuntu
wsl --update
```
Typical causes are an unstable internet connection or antivirus/firewall interference.

> **Never run `wsl --unregister Ubuntu` once you have GROMACS or project files in it.** It deletes everything permanently.

**The Ubuntu window stays black, or `wsl --shutdown` itself hangs.** The WSL background processes are stuck.
1. Wait 2–3 minutes and close most Chrome tabs. Low memory makes it worse.
2. Open Task Manager (Ctrl+Shift+Esc) → **Details** tab. End the tasks **`vmmem`** (or `vmmemWSL`), **`wsl.exe`** and **`wslservice.exe`**. This is what fixed it for us.
3. Open a new PowerShell and run `wsl --shutdown`, then reopen Ubuntu.
4. If that fails, run PowerShell as Administrator and use `Restart-Service LxssManager`. As a last resort, reboot Windows. Your files are safe either way.

**"Temporary failure in name resolution"** (Ubuntu cannot find websites). Run these inside Ubuntu:
```
ping -c 2 8.8.8.8
ping -c 2 google.com
```
If the first works and the second fails, it is DNS:
```
sudo rm -f /etc/resolv.conf
echo "nameserver 8.8.8.8" | sudo tee /etc/resolv.conf
echo "nameserver 1.1.1.1" | sudo tee -a /etc/resolv.conf
```
WSL can overwrite this file when it restarts. Make it permanent by first checking `cat /etc/wsl.conf`. If the file does not exist, create it with `printf "[network]\ngenerateResolvConf = false\n" | sudo tee /etc/wsl.conf`, then run `wsl --shutdown` from PowerShell. If it already has content, edit it by hand instead of overwriting it.

---
## Part 2 - Ubuntu: install GROMACS and tools

*Run in: the Ubuntu terminal.*

```
sudo apt update && sudo apt upgrade -y
sudo apt install -y gcc cmake build-essential libfftw3-dev
sudo apt install -y gromacs
gmx --version
```

Useful extras:
```
sudo apt-get install -y pymol grace unzip perl gedit
```
- **PyMOL** and **Grace** (`xmgrace`) are for viewing structures and plotting `.xvg` files.
- If `gedit` does not open a window, use `nano filename` instead (Ctrl+O saves, Ctrl+X exits).
- **Chimera** is downloaded from https://www.cgl.ucsf.edu/chimera/download.html. Then run `chmod +x <installer>.bin` and `./<installer>.bin`. In our workflow Chimera is mostly used on Windows for preparing the ligand and receptor.

**You install GROMACS once.** It is still there every time you reopen Ubuntu.

> **About CUDA.** CUDA only works with **NVIDIA** GPUs. A laptop with Intel or AMD graphics (ours had Intel HD 620 and an AMD FirePro) cannot use it. That is fine, because every local step in this guide is CPU work. The GPU-accelerated build of GROMACS is compiled on Colab in Part 7.

---
## Part 3 - Create a project folder and move files in

*Run in: the Ubuntu terminal.*

```
mkdir ~/gromacs_project
cd ~/gromacs_project
pwd
```
Always `cd` into the project folder before running `gmx` commands, because GROMACS reads and writes in whatever folder you are in. Work inside the Linux folder (`~/...`), not directly under `/mnt/c/...`, which is much slower.

### Get your files from Windows into Ubuntu
**Recommended:** copy from inside Ubuntu. This keeps the correct ownership and does not create extra files:
```
cp /mnt/c/Users/<WindowsUser>/Desktop/MyFolder/REC.pdb ~/gromacs_project/
cp /mnt/c/Users/<WindowsUser>/Desktop/MyFolder/LIG.mol2 ~/gromacs_project/
ls -lh
```
Alternatively, `explorer.exe .` opens the current folder in Windows File Explorer, where you can drag and drop files. Your Windows Desktop may live inside OneDrive. If a path fails, check with `ls /mnt/c/Users/<WindowsUser>/`.

### Problem: extra `...:Zone.Identifier` files
Drag-and-drop from Windows creates harmless metadata files named like `LIG.mol2:Zone.Identifier`. Delete them:
```
find . -name "*:Zone.Identifier" -delete
```

### Problem: "Permission denied" when running a script
```
sudo chown -R $USER:$USER ~/gromacs_project
chmod -R u+rwX ~/gromacs_project
chmod +x sort_mol2_bonds.pl
```

### Terminal habits that save time
- **Paste one line at a time.** Several lines pasted together often get merged or garbled.
- **Do not click or select text in the Ubuntu window while a long job runs.** Selecting text pauses the output (the title bar shows "Select ..."). Press **Esc** to continue.
- Before closing Ubuntu, check that no command is still running (you should see your prompt).

---
## Part 4 - Prepare the ligand and the receptor

*Run in: Windows (Chimera, a browser, a text editor).*

1. In **Chimera**, open the best docking pose (ligand plus receptor). Delete the protein chain so only the ligand remains, **add hydrogens**, and save as `LIG.mol2`.
2. Open `LIG.mol2` in a text editor and fix it:
   - `@<TRIPOS>MOLECULE` must be the **first line**. Delete any header or blank lines above it.
   - The name on the line after it must be **LIG**.
   - The `@<TRIPOS>BOND` section must have its bonds in a consistent order. Fix this with the Perl script, using a *different* output name (the same file as input and output can fail):
```
perl sort_mol2_bonds.pl LIG.mol2 LIG_sorted.mol2
mv LIG_sorted.mol2 LIG.mol2
```
3. Go to **SwissParam** (http://www.swissparam.ch/), upload `LIG.mol2`, choose the **CHARMM** force field, and download the `.zip` of results. Unzip it with `unzip`.
4. Open the docking pose again in Chimera, delete the ligand, run **Dock Prep** on the protein, and save it as `REC.pdb`.
5. Make one working folder and put in it: the SwissParam files (`LIG.itp`, `LIG.pdb`, `LIG.par` and so on), `REC.pdb`, and all your `.mdp` files (`ions.mdp`, `EM.mdp`, `NVT.mdp`, `NPT.mdp`, `MD.mdp`).

**Match the force field.** The SwissParam parameters must match the force field you choose in `pdb2gmx` (here CHARMM with CHARMM27).

---
## Part 5 - Build the system, add ions and minimize

*Run in: the Ubuntu terminal, in the working folder.*

### 5.1 Protein topology
```
gmx pdb2gmx -f REC.pdb -ignh
```
When asked, pick the **CHARMM27** force field (entry **8** in our version) and the **TIP3P** water model (entry **1**). Check the names, because the numbers can differ between versions. This creates `conf.gro`, `topol.top` and `posre.itp`.

### 5.2 Convert the ligand and merge it into `conf.gro`
```
gmx editconf -f LIG.pdb -o LIG.gro
```
1. Open both `.gro` files. Copy the **atom lines** of `LIG.gro` (from line 3 down to the second-to-last line, so *not* the title, the atom count, or the box line) and paste them into `conf.gro` just above its last line (the box vectors).
2. Line 2 of `conf.gro` holds the total atom count. It must equal the protein atoms plus the ligand atoms. This command prints the correct number (all lines minus the title, count and box lines):
```
awk 'END{print NR-3}' conf.gro
```
3. Put that number on line 2. Open the file in Chimera to check that the ligand and receptor both appear.

### 5.3 Edit `topol.top`
Add the ligand include **below the force-field include line** (for CHARMM27 that is `#include "charmm27.ff/forcefield.itp"`):
```
; Include ligand topology
#include "LIG.itp"
```
At the very bottom, add the ligand under the protein line in `[ molecules ]`:
```
[ molecules ]
; Compound        #mols
Protein_chain_X     1
LIG                 1
```
Also open `LIG.itp` and make sure the molecule name under `[ moleculetype ]` is `LIG`:
```
[ moleculetype ]
; Name   nrexcl
LIG      3
```

> **Check the end of `topol.top` before solvating.** If the last line has no line break, `gmx solvate` can glue its `SOL` entry onto the end of the previous line, giving something like `LIG                 1SOL             17625`. GROMACS then ignores the water, and `grompp` fails later with "number of coordinates in coordinate file ... does not match topology". Look at the end with `tail -5 topol.top`, and put every molecule on its own line.

### 5.4 Box, water and ions
```
gmx editconf -f conf.gro -d 1.0 -bt triclinic -o box.gro
gmx solvate -cp box.gro -cs spc216.gro -p topol.top -o box_sol.gro
tail -8 topol.top
```
The `tail` should now show `SOL` with a count on its own line. Then prepare the ion step:
```
gmx grompp -f ions.mdp -c box_sol.gro -p topol.top -o ION.tpr
```
A **NOTE** saying "System has non-zero total charge" is expected here. The next command neutralizes it. (Use `-maxwarn 2` only if you understand each warning you are overriding.)
```
gmx genion -s ION.tpr -p topol.top -conc 0.1 -neutral -o box_sol_ion.gro
```
When asked which group to replace, choose the group named **SOL** (it was entry 15 in the original tutorial, but read the list). This adds 0.1 M salt and neutralizes the charge.

### 5.5 Energy minimization
```
gmx grompp -f EM.mdp -c box_sol_ion.gro -p topol.top -o EM.tpr
gmx mdrun -v -deffnm EM
tail -20 EM.log
```
In `EM.log`, confirm that minimization **converged** (the maximum force fell below the `emtol` value in `EM.mdp`) before moving on.

---
## Part 6 - Ligand restraints and index files (needed for equilibration)

*Run in: the Ubuntu terminal.*

### 6.1 Position restraints for the ligand
```
gmx make_ndx -f LIG.gro -o index_LIG.ndx
```
At the `>` prompt type `0 & ! a H*` and press Enter (all non-hydrogen atoms), then `q`. Next:
```
gmx genrestr -f LIG.gro -n index_LIG.ndx -o posre_LIG.itp -fc 1000 1000 1000
```
Select the group you just created (the non-hydrogen group; it was group 3 in the tutorial). Then add this to the **end** of `topol.top`, after the protein position-restraint include:
```
; Ligand position restraints
#ifdef POSRES
#include "posre_LIG.itp"
#endif
```

### 6.2 System index file
```
gmx make_ndx -f EM.gro -o index.ndx
```
At the `>` prompt type `1 | 13` (Protein OR LIG, using *your* numbers for Protein and LIG), then `q`. This creates a combined **Protein_LIG** group for thermostat coupling.

### 6.3 Before leaving for Colab, gather these files
`EM.gro`, `topol.top`, `LIG.itp`, `posre.itp`, `posre_LIG.itp`, `index.ndx`, `NVT.mdp`, `NPT.mdp`, `MD.mdp`. Put them in one folder, zip it, and upload it to Google Drive (for example `MyDrive/COX-diclo`).

---
## Part 7 - Google Colab: GPU runtime and a CUDA-enabled GROMACS

*Run in: Google Colab. The code cells below are runnable.*

### 7.1 Pick the runtime
**Runtime → Change runtime type**: choose a **GPU**, and turn on **High-RAM** if offered.
- A **T4 or L4** is plenty for a system of about 60,000 atoms. We ran on an L4 at about 161 ns/day.
- An A100/H100 costs more compute units per hour and gains little at this system size.
- **TPUs are not supported** by GROMACS.

> Changing the runtime type (or any disconnect) gives you a **brand-new virtual machine**. Everything installed or saved on the VM is gone. Only files in Google Drive survive. You will repeat the compile step in each new session unless you cache the build (see 7.5).

In [ ]:
# 7.2 Mount Google Drive (authorize when asked)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Confirm a GPU is attached (look for "T4" or "L4")
!nvidia-smi

### 7.3 Compile GROMACS with CUDA
The `apt install gromacs` version is **CPU-only**, even on a GPU runtime. To use the GPU, compile from source. This usually takes **15–30 minutes or more**. The `&&` chain stops at the first error, so a failed `cmake` does not run `make`.

In [ ]:
%cd /content
!wget -q https://ftp.gromacs.org/gromacs/gromacs-2025.3.tar.gz
!tar xfz gromacs-2025.3.tar.gz
%cd /content/gromacs-2025.3
!rm -rf build
!mkdir build
%cd /content/gromacs-2025.3/build
!cmake .. -DGMX_BUILD_OWN_FFTW=ON -DREGRESSIONTEST_DOWNLOAD=OFF -DGMX_GPU=CUDA -DCMAKE_INSTALL_PREFIX=/usr/local/gromacs && make -j$(nproc) && sudo make install

In [ ]:
# 7.4 Make `gmx` available to every later cell and confirm CUDA support
import os
os.environ["PATH"] = "/usr/local/gromacs/bin:" + os.environ["PATH"]
!gmx --version | grep -i -E "GROMACS version|GPU support"

The last line should read `GPU support: CUDA`. If it says `disabled`, the build did not use CUDA.

**If `cmake` fails with an error like this** (we hit it on an L4 GPU):
```
Checking if nvcc accepts flags --generate-code=arch=compute_89,code=sm_89 - No
Your choice of 89 in GMX_CUDA_TARGET_SM was not accepted by nvcc
```
the CUDA compiler installed on the VM is not accepting the architecture code for your GPU. Colab's software images change over time, so a recipe that worked for months can break. Things to try, in order:
1. Look at what is installed with `!nvcc --version` and `!nvidia-smi`.
2. Always delete the old build folder first (`rm -rf build`). A stale cache causes confusing repeats of the same error.
3. Re-run `cmake` with a lower, widely supported target by adding `-DGMX_CUDA_TARGET_SM=80`. This should still run on newer GPUs, with possibly slightly less optimization, so check the ns/day afterwards.
4. Try a different GROMACS release, or a different GPU type from the runtime menu.

### 7.5 Optional: cache the compiled build in Drive
*We have not verified this on every Colab image. If it fails, just recompile.*

In [ ]:
# Save the build once (after a successful compile)
!tar czf /content/drive/MyDrive/gromacs_cuda_build.tar.gz -C /usr/local gromacs

In [ ]:
# In a later session: restore it instead of compiling again
!tar xzf /content/drive/MyDrive/gromacs_cuda_build.tar.gz -C /usr/local
import os
os.environ["PATH"] = "/usr/local/gromacs/bin:" + os.environ["PATH"]
!gmx --version | grep -i -E "GROMACS version|GPU support"

---
## Part 8 - NVT and NPT equilibration (Colab)

Move into the Drive folder that holds your uploaded files, and check they are all there:

In [ ]:
%cd /content/drive/MyDrive/COX-diclo
!ls

### 8.1 NVT (constant temperature)
`grompp` only builds the input file, so it takes seconds. `mdrun` does the real work.

In [ ]:
!gmx grompp -f NVT.mdp -c EM.gro -r EM.gro -p topol.top -n index.ndx -maxwarn 2 -o NVT.tpr
!gmx mdrun -v -deffnm NVT -nb gpu -pme gpu -bonded gpu -update gpu

For us, 100 ps took about 94 s on an L4. In the log, look for lines such as "PME tasks will do all aspects on the GPU", which show the GPU is really being used. Optionally confirm the temperature settled near your target:

In [ ]:
!echo "Temperature 0" | gmx energy -f NVT.edr -o NVT_temperature.xvg

*(If the name is not accepted, run `gmx energy` interactively and type the number shown next to Temperature, then `0`.)*

### 8.2 NPT (constant pressure)

In [ ]:
!gmx grompp -f NPT.mdp -c NVT.gro -r NVT.gro -p topol.top -n index.ndx -maxwarn 2 -o NPT.tpr
!gmx mdrun -v -deffnm NPT -nb gpu -pme gpu -bonded gpu -update gpu

In [ ]:
# Pressure and density should fluctuate around stable values
!echo "Pressure Density 0" | gmx energy -f NPT.edr -o NPT_pressure_density.xvg

---
## Part 9 — Production MD (100 ns) without losing your data

This part exists because we lost track of a trajectory once. Read the lessons before you start.

### Lessons learned
1. **Colab runs one cell at a time.** A running `mdrun` cell blocks every other cell, which just sit in a queue. The fix is to start `mdrun` **in the background** (below) so the cell returns immediately. The Colab **Terminal** button (bottom left) also gives you a separate shell for checking on the run.
2. **Check that `MD.xtc` really exists and is growing**, in the first 20–30 minutes and again after a few hours. A run can look perfectly healthy while its trajectory file is not where you expect.
3. **Checkpoints are not trajectories.** `MD.cpt` only lets the run *resume*. It does not contain the frames you will analyse.
4. **Sessions disconnect.** Colab Pro+ "background execution" is useful but not a guarantee. Plan to resume from a checkpoint.
5. Write to Google Drive so your files outlive the VM, but verify them (point 2).

### 9.1 Build the production input
```
gmx grompp -f MD.mdp -c NPT.gro -t NPT.cpt -p topol.top -n index.ndx -maxwarn 2 -o MD.tpr
```
(`-t NPT.cpt` carries over the exact final state of NPT. Edit the run length in `MD.mdp` first. For 100 ns at `dt = 0.002` ps, `nsteps = 50000000`. Frames are written every `nstxout-compressed` steps, so a value of 5000 gives one frame per 10 ps.)

In [ ]:
%cd /content/drive/MyDrive/COX-diclo
!gmx grompp -f MD.mdp -c NPT.gro -t NPT.cpt -p topol.top -n index.ndx -maxwarn 2 -o MD.tpr

#First, run grompp to generate your binary input file (md_prod.tpr)

In [ ]:
!gmx grompp -f MD.mdp -c NPT.gro -t NPT.cpt -p topol.top -n index.ndx -o md_prod.tpr

### 9.3 Start the real MD run

In [ ]:
!gmx mdrun -deffnm md_prod -cpt 10 -v

### 9.4 Resuming from your last MD run
Incase of power failure or netwrork glitches, the Colab is going to stop. So you will need to re-run this code in order to resume to your Colab

In [ ]:
%%bash
!cd /content/drive/MyDrive/EGFR-sil

# Resume simulation using the existing checkpoint file
!gmx mdrun -deffnm md_prod -cpi md_prod.cpt -cpt 10 -v

### 9.5 If the session disconnects: resume from the checkpoint
You may see "Resuming execution" and lose the VM. Reconnect, mount Drive, restore or recompile GROMACS (Part 7), `cd` into the Drive folder, then:

In [ ]:
%%bash
!cd /content/drive/MyDrive/EGFR-sil

# -noappend tells GROMACS to start a new output file set rather than failing on log checksums
!gmx mdrun -deffnm md_prod -cpi md_prod.cpt -noappend -cpt 10 -v

**What to expect, and what went wrong for us**
- If the run is resumed and GROMACS cannot find the earlier output files to append to, it starts new **part files** such as `MD.part0002.xtc`, `MD.part0002.log` and `MD.part0002.edr`. This is normal. You merge them at the end (9.7).
- If `mdrun` stops with *"Some output files listed in the checkpoint file are not present or not named as the output files"*, first run `ls -la MD.part000*` to see what is really there. In our case the files were present but had not appeared yet in the freshly mounted Drive, and **simply retrying a few minutes later worked**. If they are truly missing, GROMACS offers a safe option: add `-noappend` to start new part files. You lose nothing.
- The Google Drive **website** can lag behind what Colab sees. If Colab lists a file that the web page does not show, refresh the page, search for the file name, or wait a few minutes. The file is not lost.
- A message such as "Saving failed ... updated remotely or in another tab" refers only to the notebook document. It does not affect the simulation.

### 9.6 Rescue: an `MD.xtc` that is missing from your Drive folder
Our `MD.xtc` was not in the Drive folder even though the run was progressing. Google Drive's sync client had stored it in a hidden `lost_and_found` folder **on the Colab VM**. This is only on that VM, so **check while the same session is still alive**:

In [ ]:
!find /root/.config/Google/DriveFS/lost_and_found/ -type f -exec ls -lh {} \;

If a trajectory file is listed, **copy it to Drive immediately** under a clear name, then check what it covers:

In [ ]:
# Replace the path with the one printed above
!cp "/root/.config/Google/DriveFS/lost_and_found/<folder-id>/MD.xtc" /content/drive/MyDrive/COX-diclo/MD_part1_recovered.xtc
!gmx check -f /content/drive/MyDrive/COX-diclo/MD_part1_recovered.xtc

In the `gmx check` output, read the **number of frames** and the **time step**, and compare the frame count with the time you expect. In our project the recovered file held 5,757 frames (about 57.6 ns at 10 ps per frame), which ended exactly where the checkpoint resume began. (We first misread a progress line in this output as the file's end and wrongly thought there was a gap. Always confirm with the frame count and time step.)

### 9.7 Merge all trajectory parts into one file
List the files in **time order**. `trjcat` keeps the original time stamps, and when two files overlap in time it uses the frame from the *later* file.

In [ ]:
%cd /content/drive/MyDrive/COX-diclo
!gmx trjcat -f MD_part1_recovered.xtc MD.part0002.xtc MD.part0003.xtc -o COX_diclo_100ns_final.xtc
!gmx check -f COX_diclo_100ns_final.xtc

Verify the merged file: the first frame should be at time 0, the last at your full run time (here **100000 ps**), and the frame count should be consistent with a uniform spacing. Ours had **10,001 frames at 10 ps**, which is exactly right for 100 ns.

**Optional (we did not need it, so it is untested here):** the energy files are split into parts too. If you want to use `gmx energy` on the whole run, merge them first with `gmx eneconv -f MD.edr MD.part0002.edr MD.part0003.edr -o MD_all.edr`.

---
## Part 10 - Bring the results back and fix periodicity

*Run in: the Ubuntu terminal.*

### 10.1 Download and copy into Ubuntu
From Google Drive download three files: the merged trajectory (`COX_diclo_100ns_final.xtc`), `MD.tpr`, and `index.ndx`. (A multi-gigabyte file triggers a virus-scan message. Click **Download anyway**.) Then:
```
mkdir -p ~/COX-diclo/analysis
cd ~/COX-diclo/analysis
cp /mnt/c/Users/<WindowsUser>/Desktop/MyFolder/COX_diclo_100ns_final.xtc .
cp /mnt/c/Users/<WindowsUser>/Desktop/MyFolder/MD.tpr .
cp /mnt/c/Users/<WindowsUser>/Desktop/MyFolder/index.ndx .
ls -lh
```
A 2 GB file takes a few minutes to copy and shows no progress, so wait for the prompt to return.

**Optional, to save time on a slow connection:** you can create a smaller, water-free trajectory on Colab first with `gmx trjconv ... -n index.ndx` and the `Protein_LIG` group, and download that instead. Keep the full system for the MM/GBSA step in Part 12, which needs a trajectory that matches `MD.tpr`.

### 10.2 Check the trajectory and list the groups
```
gmx check -f COX_diclo_100ns_final.xtc
gmx make_ndx -f MD.tpr -n index.ndx
```
At the `>` prompt type `q` to quit without changing anything. Our groups looked like this:

| No. | Group | | No. | Group |
|---|---|---|---|---|
| 0 | System | | 13 | LIG |
| 1 | Protein | | 14 / 15 | NA / CL |
| 3 | C-alpha | | 16 / 17 | Water / SOL |
| 4 | Backbone | | 20 | Water_and_ions |
| | | | 21 | Protein_LIG |

**Your numbers will differ.** Use the ones from your own list in every command below.

### 10.3 Fix periodicity (do not skip this)
Raw trajectories let molecules cross the box boundary, which gives jumpy and wrong RMSD values. Make the molecules whole and centered, then remove rotation and translation. Using `echo` supplies the group answers (centering group, then output group) so each step is a single line.
```
echo "1 0" | gmx trjconv -s MD.tpr -f COX_diclo_100ns_final.xtc -n index.ndx -o noPBC.xtc -pbc mol -ur compact -center
echo "4 0" | gmx trjconv -s MD.tpr -f noPBC.xtc -n index.ndx -o fit.xtc -fit rot+trans
gmx check -f fit.xtc
```
- Step 1: center on **Protein** (1), output **System** (0). `-ur compact` is the form recommended for the triclinic box we built in Part 5.
- Step 2: fit on **Backbone** (4), output **System** (0).
- The final check should still show your full frame count and end time.

Each step reads and writes about 2 GB and takes a few minutes.

---
## Part 11 — Analysis: RMSD, RMSF, Rg, hydrogen bonds, SASA

*Run in: the Ubuntu terminal, in `~/COX-diclo/analysis`.*

Each command writes an `.xvg` file. The `echo` supplies the group number(s) at the prompts. Remember to substitute your own numbers.

```
# RMSD of the protein backbone (fit on Backbone, calculate on Backbone)
echo "4 4" | gmx rms -s MD.tpr -f fit.xtc -n index.ndx -o rmsd_backbone.xvg -tu ns

# RMSD of the ligand (fit on Backbone, calculate on LIG)
echo "4 13" | gmx rms -s MD.tpr -f fit.xtc -n index.ndx -o rmsd_ligand.xvg -tu ns

# RMSF per residue (C-alpha)
echo "3" | gmx rmsf -s MD.tpr -f fit.xtc -n index.ndx -o rmsf.xvg -res

# Radius of gyration (Protein)
echo "1" | gmx gyrate -s MD.tpr -f fit.xtc -n index.ndx -o gyrate.xvg

# Hydrogen bonds between protein and ligand
gmx hbond -s MD.tpr -f fit.xtc -n index.ndx -r 'group "Protein"' -t 'group "LIG"' -num hbnum.xvg -tu ns

# Solvent-accessible surface area of the protein
gmx sasa -s MD.tpr -f fit.xtc -n index.ndx -surface 'group "Protein"' -o sasa.xvg -tu ns
```
If the `hbond` or `sasa` selection syntax is rejected, run the command **without** the `-r`/`-t` (or `-surface`) part. GROMACS then asks you to choose the groups interactively (for H-bonds: Protein, then LIG). The `gmx energy` command (using `MD.edr`) gives temperature, pressure, potential energy and so on.

### Reading the results
- **RMSD** that levels off into a plateau means the system has equilibrated. Use that plateau window for MM/GBSA.
- **RMSF** shows which residues are flexible (loops) and which are rigid (the binding site, if the ligand is stable).
- **Rg and SASA** show compactness and overall stability. Large drifts suggest unfolding or opening.
- **H-bond count** shows how persistently the ligand is held.

### Plot everything (Python, runs in Jupyter or Colab)
Put the `.xvg` files in the same folder as this notebook (or upload them to Colab) and run:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def load_xvg(path):
    # Read an .xvg file and return x, y (first two columns) plus axis labels
    x, y = [], []
    xlabel = ylabel = ""
    with open(path) as f:
        for line in f:
            if line.startswith("#"):
                continue
            if line.startswith("@"):
                if "xaxis" in line and "label" in line:
                    xlabel = line.split('"')[1]
                elif "yaxis" in line and "label" in line:
                    ylabel = line.split('"')[1]
                continue
            parts = line.split()
            if len(parts) >= 2:
                x.append(float(parts[0]))
                y.append(float(parts[1]))
    return np.array(x), np.array(y), xlabel, ylabel

def panel(ax, files, title):
    plotted = False
    for path, label in files:
        try:
            x, y, xl, yl = load_xvg(path)
        except FileNotFoundError:
            print("skipping (not found):", path)
            continue
        ax.plot(x, y, lw=1, label=label)
        ax.set_xlabel(xl or "x")
        ax.set_ylabel(yl or "y")
        plotted = True
    ax.set_title(title)
    if plotted:
        ax.legend(fontsize=8)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
panel(axes[0, 0], [("rmsd_backbone.xvg", "Backbone"), ("rmsd_ligand.xvg", "Ligand")], "RMSD")
panel(axes[0, 1], [("rmsf.xvg", "C-alpha")], "RMSF")
panel(axes[0, 2], [("gyrate.xvg", "Protein")], "Radius of gyration")
panel(axes[1, 0], [("hbnum.xvg", "Protein–LIG")], "Hydrogen bonds")
panel(axes[1, 1], [("sasa.xvg", "Protein")], "SASA")
axes[1, 2].axis("off")
plt.tight_layout()
plt.savefig("md_analysis_summary.png", dpi=300)
plt.show()

---
## Part 12 - MM/GBSA binding energy with gmx_MMPBSA

*Run in: the Ubuntu terminal.* MM/GBSA is much faster than MM/PBSA and suits a laptop. On our 2-core laptop, 201 frames took about **59 minutes**.

### 12.1 Install (once)
Install Miniconda, then reopen Ubuntu. On a slow or unstable connection, `-c` makes `wget` resume and retry:
```
wget -c --tries=20 --retry-connrefused --waitretry=5 https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
bash Miniconda3-latest-Linux-x86_64.sh
```
Accept the defaults and say `yes` to initialize conda. Close and reopen Ubuntu (the prompt should start with `(base)`), then create the environment. The documented route is a conda environment containing AmberTools, followed by a pip install of gmx_MMPBSA.
```
conda create -n gmxMMPBSA -c conda-forge --override-channels python=3.11.8 -y
conda activate gmxMMPBSA
conda install -c conda-forge --override-channels "mpi4py=4.0.1" "ambertools<=23.3" -y
conda install -c conda-forge --override-channels "numpy=1.26.4" "matplotlib=3.7.3" "scipy=1.14.1" "pandas=1.5.3" "seaborn=0.11.2" -y
python -m pip install gmx_MMPBSA
gmx_MMPBSA --help
```
The AmberTools download is several GB and can take a long time on a weak connection. Plug the laptop in and stop it sleeping. If conda prints "Network is unreachable" or timeout errors, it is retrying. If it stalls, press Ctrl+C, make conda more patient, and rerun the same install command (finished downloads are kept):
```
conda config --set remote_read_timeout_secs 180
conda config --set remote_connect_timeout_secs 30
conda config --set remote_max_retries 10
conda config --set fetch_threads 2
```
If conda stops with a "Terms of Service" message, accept them with `conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main` (and the same for `.../pkgs/r`).

> **Do not install GROMACS inside this conda environment.** An older conda GROMACS cannot read a `.tpr` written by a newer GROMACS (ours came from 2025.3). Leave it out, so gmx_MMPBSA uses your system `gmx`.

### 12.2 Prepare the inputs
Copy the topology files next to your trajectory:
```
cd ~/COX-diclo/analysis
cp ~/COX-diclo/topol.top ~/COX-diclo/*.itp .
```
Cut a manageable number of frames from the **stable** part of the run (decide from your RMSD plot). We took the last 20 ns, one frame every 100 ps, and kept the full system so it matches `MD.tpr`:
```
echo "0" | gmx trjconv -s MD.tpr -f fit.xtc -n index.ndx -o mmpbsa.xtc -b 80000 -e 100000 -dt 100
```
(`-b`, `-e` and `-dt` are in ps. This gave 201 frames.)

The input file, saved as `mmgbsa_test.in`, can be created with this single line:
```
printf '&general\nsys_name="COX_diclo_GB",\nstartframe=1,\nendframe=10,\ninterval=1,\ntemperature=300,\nverbose=2,\n/\n&gb\nigb=5, saltcon=0.150,\n/\n' > mmgbsa_test.in
```
It contains:
```
&general
  sys_name="COX_diclo_GB",
  startframe=1,
  endframe=10,
  interval=1,
  temperature=300,
  verbose=2,
/
&gb
  igb=5, saltcon=0.150,
/
```

### 12.3 Test on 10 frames first
The receptor group is **Protein** and the ligand group is **LIG** (zero-based numbers from your index, here `1 13`):
```
gmx_MMPBSA -O -i mmgbsa_test.in -cs MD.tpr -ci index.ndx -cg 1 13 -ct mmpbsa.xtc -cp topol.top -o TEST_RESULTS.dat -eo TEST_energy.csv -nogui
tail -40 TEST_RESULTS.dat
```
This checks that your CHARMM topology and ligand parameters load correctly, which is where most problems appear. The bonded terms (ΔBOND, ΔANGLE, ΔDIHED) should be about zero. Ten frames is too few to trust the numbers.

### 12.4 Full run
```
sed 's/endframe=10,/endframe=201,/' mmgbsa_test.in > mmgbsa.in
grep endframe mmgbsa.in
gmx_MMPBSA -O -i mmgbsa.in -cs MD.tpr -ci index.ndx -cg 1 13 -ct mmpbsa.xtc -cp topol.top -o FINAL_RESULTS_MMGBSA.dat -eo FINAL_RESULTS_MMGBSA.csv -nogui
```
Keep Ubuntu open, the laptop plugged in, and other heavy programs closed (8 GB of RAM is tight). **Do not select text in the window**, as it pauses the progress. The run goes through *Complex*, then *Receptor*, then *Ligand* stages, and the receptor stage is the slowest. When it ends it reports "Run completed with 0 errors and N warnings".

### 12.5 Read, report and tidy
```
tail -45 FINAL_RESULTS_MMGBSA.dat
```
In the **Delta (Complex − Receptor − Ligand)** block, report **ΔTOTAL** as mean ± SD, together with its components (ΔVDWAALS, ΔEEL, ΔEGB, ΔESURF). The `.csv` file holds the per-frame energies, so you can also plot binding energy against time. Save your results and clean up the temporary files:
```
gmx_MMPBSA --clean
```

**About the warnings.** With a CHARMM topology, gmx_MMPBSA warns that the stored radii (mbondi3) differ from what `igb=5` is conventionally paired with (mbondi2), and that GB with CHARMM topologies is a cross-parameterized protocol. The run still keeps the radii stored in your topology and is valid, but **state this in your methods**. Per the Amber manual, `igb=8` pairs with mbondi3, so reviewers may ask whether you tested it. Confirm in the manual before relying on that, then repeat the run with `igb=8` if needed.

**How to interpret the number**
- It excludes entropy, so call it an *estimated* binding free energy.
- MM/GBSA values are usually more negative than experimental affinities. Use them to **compare ligands**, not to match an experimental number.
- Negative ΔTOTAL means favorable binding. Check whether van der Waals or electrostatic terms dominate.

---
## Part 13 - Quick troubleshooting table

| What you see | Likely cause | What to do |
|---|---|---|
| `WslRegisterDistribution ... 0x80370102` | Virtualization off | Part 1 (Windows features, then BIOS) |
| "Waiting for OOBE command to complete" | First-run setup hung | Shutdown, unregister (only if nothing is saved), reinstall (Part 1) |
| Ubuntu black; `wsl --shutdown` hangs | Stuck WSL processes | End `vmmem`, `wsl.exe`, `wslservice.exe` in Task Manager |
| "Temporary failure in name resolution" | DNS in WSL | Set nameservers in `/etc/resolv.conf` (Part 1) |
| `...:Zone.Identifier` files | Windows download marker | `find . -name "*:Zone.Identifier" -delete` |
| "Permission denied" on a script | Ownership or permissions | `chown`, `chmod +x`; use a different output filename |
| `grompp`: coordinates do not match topology | `[ molecules ]` wrong, e.g. `LIG 1SOL n` on one line | Put each molecule on its own line (Part 5.3) |
| NOTE: non-zero total charge at the ions step | Expected | `genion -neutral` fixes it |
| Pasted commands run together | Multi-line paste | Paste one line at a time |
| Output frozen, title bar says "Select" | Text selected in console | Press Esc |
| `cmake`: nvcc does not accept `compute_89` | CUDA toolkit and GPU architecture mismatch, or stale build | Part 7.4 steps |
| Colab cells stay queued | One cell runs at a time | Run `mdrun` with `nohup ... &` or use the Terminal |
| `MD.xtc` not in the Drive folder | Drive sync parked it in `lost_and_found` | Part 9.6, while the session is alive |
| New files named `MD.part0002.xtc` | Resume could not append | Normal; merge with `trjcat` (Part 9.7) |
| "output files listed in the checkpoint ... not present" | Files not visible yet, or renamed | Wait and retry; `ls`; or add `-noappend` |
| "Resuming execution", files on the VM gone | New VM after a disconnect | Reinstall GROMACS; resume with `-cpi` from Drive |
| Drive website misses a file Colab shows | Web view lag | Refresh, search by name, wait |
| RMSD looks jumpy or wrong | Periodicity not fixed | `trjconv -pbc mol -center`, then fit (Part 10.3) |
| conda download timeouts | Unstable network | `conda config` timeouts, rerun (Part 12.1) |
| gmx_MMPBSA cannot read `MD.tpr` | Older GROMACS in the environment | Do not install conda GROMACS |
| gmx_MMPBSA radii warnings (mbondi3 / `igb=5`) | CHARMM topology cross-parameterization | Disclose; optionally test `igb=8` |

---
## Part 14 - Checklists and a methods template

### Before you start production
- [ ] EM converged (Part 5.5), and NVT and NPT finished without errors.
- [ ] `MD.mdp` has the run length you want and a sensible `nstxout-compressed`.
- [ ] A 10,000-step test run wrote a real `MD_test.xtc`.
- [ ] `mdrun` is started in the background (`nohup ... &`) with `-cpt 10`.
- [ ] `MD.xtc` exists after about 20–30 minutes and its size is growing.

### After production
- [ ] Merge any part files and run `gmx check` (first frame at 0, last at your full time).
- [ ] Download the trajectory, `MD.tpr` and `index.ndx`, and **back them up** (a second copy on another drive).
- [ ] Fix periodicity before any analysis.
- [ ] Save plots, `.xvg` files and the MM/GBSA `.dat` and `.csv` files, and keep `gmx_MMPBSA.log` as a record of the settings.

### Methods paragraph template (replace the [brackets])
*The protein–ligand complex was prepared using [UCSF Chimera] and [SwissParam] ligand parameters, and simulated with GROMACS [2025.3] using the CHARMM27 force field and the TIP3P water model in a triclinic box with a [1.0] nm solute–box distance. The system was neutralized with counter-ions and brought to [0.1] M salt. After energy minimization, the system was equilibrated for [100] ps in the NVT ensemble at [300] K and for [X] ps in the NPT ensemble, followed by a [100] ns production run with a [2] fs time step, saving coordinates every [10] ps. Production was run on [an NVIDIA L4 GPU] and resumed from checkpoints where needed; trajectory parts were concatenated into a single continuous trajectory. Trajectories were centered and fitted on the protein backbone, and RMSD, RMSF, radius of gyration, hydrogen bonds and SASA were calculated with GROMACS tools. Binding free energy was estimated with gmx_MMPBSA ([MM/GBSA, igb=5, 0.150 M salt]) on [201] frames from [80–100] ns, excluding entropy.*

### Citations to include
- Valdés-Tresanco, M.S., Valdés-Tresanco, M.E., Valiente, P.A. and Moreno, E. gmx_MMPBSA: A New Tool to Perform End-State Free Energy Calculations with GROMACS. *J. Chem. Theory Comput.* 2021, 17(10), 6281–6291.
- Miller III, B.R., McGee Jr., T.D., Swails, J.M., Homeyer, N., Gohlke, H. and Roitberg, A.E. MMPBSA.py: An Efficient Program for End-State Free Energy Calculations. *J. Chem. Theory Comput.* 2012, 8(9), 3314–3321.
- The GROMACS reference papers for the version you used, and the SwissParam publication.

### Useful links
- GROMACS documentation: https://manual.gromacs.org
- gmx_MMPBSA documentation: https://valdes-tresanco-ms.github.io/gmx_MMPBSA/
- SwissParam: http://www.swissparam.ch/